# Polymorphism

Polymorphism means "many forms." It allows different classes to have methods with the same name but different behaviors.

---

# 1. Duck Typing (Implicit Polymorphism)

In strictly typed languages (like Java or C++), objects have a rigid Identity. If you write a function that processes a Student, you must explicitly tell the compiler: "This function only accepts an object whose literal identity is Student."

**C++ Example:**

```cpp
#include <iostream>
using namespace std;

class Student {};

// cpp requires explicit name of object 
void display(Student obj) {}

int main(){ return 0;}
```

**Duck typing solves this rigidity**. In Python, functions don’t ask for an object's ID badge (its Class). They only ask for its Capabilities (its methods). If the object has the required capability, Python lets it pass.

In [17]:
# Student Class
class Student:
    def __init__(self, id, name, dept):
        self.id = id
        self.name = name
        self.dept = dept

    # generator (getter)
    def get(self):
        yield self.id
        yield self.name
        yield self.dept

    # setter
    def update(self, id=None, name=None, dept=None):
        self.id = id if id is not None else self.id
        self.name = name if name is not None else self.name
        self.dept = dept if dept is not None else self.dept

# Server Rack Class
class ServerRack:
    def __init__(self, rack_id, model, status):
        self.rack_id = rack_id
        self.model = model
        self.status = status

    # generator
    def get(self):
        yield self.rack_id
        yield self.model
        yield self.status

In [18]:
# Notice: we are defining single function for two class to handle which is not possible in other languages. Python resolves it based on current object and attribute/method of that class, if it matches "done"
def display(cls_obj):
    for attribute in cls_obj.get():
        print(attribute)

In [19]:
import uuid

s1 = Student(str(uuid.uuid4()), "Saqib Bedar", "Computer Science")
server = ServerRack(str(uuid.uuid4()), "Dell", "Active")

# Both work perfectly in the same function
display(s1)
print("-"*36)
display(server)

c0235c86-05b9-4055-a309-0bc9adf2482b
Saqib Bedar
Computer Science
------------------------------------
6c1c707f-4dda-455a-af63-7163dadf38df
Dell
Active


> The Purpose: High reusability. You don't need a massive, tangled inheritance tree where `ServerRack` and `Student` inherit from some weird display base class. You just give them the same method name, and Python handles the rest.

---

# Implementation Scenario 2 - Built-in Protocols (Magic Methods)

The most common way duck typing is used in Python is through "Magic Methods" (methods wrapped in double underscores, like `__ init __`). Python itself uses duck typing under the hood.

Right now, if you try to use a standard for loop on your Student object, it will fail because Python doesn't know how to iterate over it natively. You have to explicitly call `s1.get()`.

```python
# This throws an error!
for item in s1:
    print(item)
```

**How to fix it using Duck Typing**:

Python expects any iterable object to have a specific behavior: an `__ iter __()` method. If we rename our `get()` method to `__ iter __`, your Student class is now officially duck-typed as an Iterable.

In [20]:
class Student:
    def __init__(self, id, name, dept):
        self.id = id
        self.name = name
        self.dept = dept

    # generator: renamed from get to __iter__
    def __iter__(self):
        yield self.id
        yield self.name
        yield self.dept

    def update(self, id=None, name=None, dept=None):
        self.id = id if id is not None else self.id
        self.name = name if name is not None else self.name
        self.dept = dept if dept is not None else self.dept

import uuid

s1 = Student(str(uuid.uuid4()), "Saqib Bedar", "Computer Science")

# Now this works perfectly! (python automatically handles iterable)
for item in s1:
    print(item)

cf8a09cd-6168-4614-9f80-a0f5d99ccc48
Saqib Bedar
Computer Science


**When is this Duck Typing?**

When Python sees the for loop, it doesn't check `isinstance(s1, list)` or `isinstance(s1, tuple)`. It silently looks for the `__ iter __` method. Because your class has it, Python treats your custom Student exactly like a built-in list or tuple.

---

# Implementation Scenario 3 - Defensive Duck Typing (EAFP)

Because duck typing assumes the object has the required methods, it will crash if you pass an object that doesn't.

In static languages, you check types before acting (Look Before You Leap). In Python duck typing, we use EAFP: *Easier to Ask for Forgiveness* than Permission.

Instead of checking the class type, we try to use the behavior and catch the failure:

In [21]:
def update_dept(entity, new_dept):
    try:
        # We assume it has a update() method
        entity.update(dept=new_dept)
        print("Department Updated!")
    except AttributeError:
        # Handle error in case if type is different or method is non-existing
        print(f"Error: {type(entity).__name__} does not support standard updates.")

# Update student1 department
update_dept(entity=s1, new_dept="CS")

# Verify student after department update
print(s1.__dict__)

Department Updated!
{'id': 'cf8a09cd-6168-4614-9f80-a0f5d99ccc48', 'name': 'Saqib Bedar', 'dept': 'CS'}


---

# Summary of What it Solves

- **Decoupling**: Your functions don't need to import or know about specific classes.

- **Flexibility**: You can drop entirely new classes into existing functions, as long as you match the method names.

- **Simplicity**: You avoid creating pointless, empty Base Classes or Interfaces just to satisfy a compiler.

---

# More Examples

In [4]:
# Disclaimer: AI generated code!

class Duck:
    def fly(self):
        print("Flap flap! The duck is airborne.")

class Airplane:
    def fly(self):
        print("Vroom! Engines roaring, the plane takes off.")

class Whale:
    def swim(self):
        print("Splashing through the ocean.")

# A function that expects ANY object capable of flying
def make_it_fly(flying_object):
    flying_object.fly()

# Test cases
duck = Duck()
plane = Airplane()
whale = Whale()

make_it_fly(duck)   # Works perfectly
make_it_fly(plane)  # Works perfectly! Python does not care that it's a vehicle.
# make_it_fly(whale) # Raises AttributeError: 'Whale' object has no attribute 'fly'

Flap flap! The duck is airborne.
Vroom! Engines roaring, the plane takes off.
